In [ ]:

import numpy as np
import os
import tables as tb
import yaml
import zmq
from basil.dut import Dut
from online_monitor.utils import utils as ou
from tjmonopix2 import utils
from tjmonopix2.analysis import analysis_utils as au
from tjmonopix2.system.tjmonopix2 import TJMonoPix2 
from tjmonopix2.scans.scan_ext_trigger import ExtTriggerScan
from tjmonopix2.system.bdaq53 import BDAQ53

In [ ]:
# Compression for data files
FILTER_RAW_DATA = tb.Filters(complib='blosc', complevel=5, fletcher32=False)
FILTER_TABLES = tb.Filters(complib='zlib', complevel=5, fletcher32=False)
# Default locations
# PROJECT_FOLDER = os.path.join(os.path.dirname(__file__), '..')
# SYSTEM_FOLDER = os.path.join(PROJECT_FOLDER, 'system')
DEFAULT_CONFIG_FILE = '/home/wiebke/git/tj-monopix2-daq/tjmonopix2/testbench.yaml' # os.path.join(PROJECT_FOLDER, 'system', 'default.cfg.yaml')
# TESTBENCH_DEFAULT_FILE = os.path.join(PROJECT_FOLDER, 'testbench.yaml')

In [ ]:
scan_configuration = {
    'start_column': 0,
    'stop_column': 224,
    'start_row': 0,
    'stop_row': 512,

    'scan_timeout': False,    # Timeout for scan after which the scan will be stopped, in seconds; if False no limit on scan time
    'max_triggers': 1000000,  # Number of maximum received triggers after stopping readout, if False no limit on received trigger

    'tot_calib_file': None    # path to ToT calibration file for charge to e⁻ conversion, if None no conversion will be done
}

In [ ]:
class TJ_Control():
    def __init__(self, config_file=None):
        if config_file is None:
                config_file = DEFAULT_CONFIG_FILE
        self.config = config_file
        with open(self.config) as f: 
            self.tab = yaml.safe_load(f) 
    def initialize(self):
        print(self.tab)
        self.scan_ext = ExtTriggerScan(scan_config = self.tab)
        self.scan_ext.init()
    def configure(self):
        self.scan_ext.configure()
    def start(self):
        self.scan_ext.start()
    def stop(self):
        self.scan_ext.stop()
        self.dut.close()
    def reconfigure(self, new_config_file=None):
        if new_config_file:
            self.config = new_config_file
        self.scan_ext.configure()


In [ ]:
TJ = TJ_Control() 

TJ.initialize()
TJ.configure()
TJ.reconfigure()
TJ.start()